# Kaggle submission driver

The final submission (four members, 0.25 each) run from this repo's code.

Attach: the competition data; `tuckerarrants/rogii-v2s` (checkpoints of the three V2 members) and the
`tuckerarrants/rogii-l2` model (the L2 member). For the code and libraries, either turn internet on (the
repo is cloned from GitHub and `segmentation_models_pytorch` / `timm` come from PyPI), or, internet off,
attach this repo as a dataset (`python scripts/make_kaggle_src.py`) and `tuckerarrants/smp-offline` (wheels).

**As a check.** Run All in the editor (not Submit): Kaggle then shows the 3 test wells outside scoring, and
the last cell compares the result with the numbers the notebook of record printed for them.

- With the L2 model attached: `CONFIG_NAME = 'final_equal_weight'`. Every value should match to about 0.001 ft.
- Without it: `CONFIG_NAME = 'check_v2_only'`, the three V2 members at 1/3 each. This differs from the saved
  submission by a third of L2's distance from the blend, so every value must be within 0.5 ft
  (see `configs/submit/check_v2_only.yaml`).

Roughly 15–30 minutes on 2× T4, most of it loading checkpoints.

In [ ]:
import glob, subprocess, sys
wheels = sorted(glob.glob('/kaggle/input/**/smp_offline_wheels/*.whl', recursive=True))
pkgs = wheels or ['segmentation_models_pytorch==0.5.0', 'timm==1.0.27']   # no wheels attached: PyPI (internet on)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *pkgs], check=True)

In [ ]:
import glob, os, subprocess, sys
hits = sorted(glob.glob('/kaggle/input/**/rogii_wellbore/__init__.py', recursive=True))
if hits:                                   # the code dataset is attached
    REPO = os.path.dirname(os.path.dirname(os.path.dirname(hits[0])))
    sys.path.insert(0, os.path.join(REPO, 'src'))
else:                                      # internet on: clone the repo
    REPO = '/kaggle/working/rogii-wellbore-2026'
    if not os.path.exists(REPO):
        subprocess.run(['git', 'clone', '-q', 'https://github.com/TuckerArrants/rogii-wellbore-2026.git', REPO], check=True)
    sys.path.insert(0, os.path.join(REPO, 'src'))
CONFIG_NAME = 'final_equal_weight'      # or 'check_v2_only' without the rogii-l2 model
CONFIG = os.path.join(REPO, f'configs/submit/{CONFIG_NAME}.yaml')
COMP = next(p for p in ['/kaggle/input/competitions/rogii-wellbore-geology-prediction',
                        '/kaggle/input/rogii-wellbore-geology-prediction'] if os.path.exists(p))

from rogii_wellbore.infer import run
sub = run(CONFIG, COMP, 'submission.csv')
sub.head()

In [ ]:
# Check against the notebook of record (only meaningful on the 3 visible test wells)
import json
from rogii_wellbore.infer import check_reference
ref = json.load(open(os.path.join(REPO, 'pipeline/03_submit/reference_rows.json')))
if CONFIG_NAME == 'final_equal_weight':
    ok = check_reference(sub, ref)
else:   # three V2 members: within a third of L2's largest distance from the saved blend
    ok = check_reference(sub, ref, row_tol=ref['v2_only_tol'], stat_tol=ref['v2_only_tol'])
print('PASS' if ok else 'FAIL')